In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_fscore_support,
    accuracy_score,
    f1_score,
    make_scorer,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.model_selection import RandomizedSearchCV

import xgboost as xgb
from tensorflow import keras
from tensorflow.keras import layers

from pathlib import Path
import joblib

artifacts_dir = Path("artifacts")
pre = joblib.load(artifacts_dir / "preprocessing_artifacts.joblib")
globals().update(pre)
print("Loaded preprocessing artifacts from", artifacts_dir / "preprocessing_artifacts.joblib")


Loaded preprocessing artifacts from artifacts\preprocessing_artifacts.joblib


## 5. Model Training

We train and evaluate five models: Logistic Regression (baseline), Random Forest, XGBoost, DNN, and a Stacking ensemble.

In [2]:
def evaluate_model(name, y_true, y_pred, y_proba=None):
    """Print classification report, confusion matrix, and optionally ROC-AUC."""
    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print("="*60)
    # Support: per-class = # samples in that class; accuracy/macro/weighted rows = total # samples (test set size)
    print(classification_report(y_true, y_pred, target_names=["Low", "High"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")
    auc = None
    if y_proba is not None and len(np.unique(y_true)) > 1:
        auc = roc_auc_score(y_true, y_proba)
        print(f"ROC-AUC: {auc:.4f}")
    p_high, r_high, f1_high, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1)
    p_low, r_low, f1_low, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=0)
    return {"accuracy": acc, "roc_auc": auc,
            "precision_high": p_high, "recall_high": r_high, "f1_high": f1_high,
            "precision_low": p_low, "recall_low": r_low, "f1_low": f1_low,
            "predictions": y_pred, "probabilities": y_proba}

### 5.1 Logistic Regression - Baseline Model

In [3]:
# Logistic Regression (baseline)
lr_simple = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_simple.fit(X_train_scaled, y_train)
y_pred_lr_simple = lr_simple.predict(X_test_scaled)
y_proba_lr_simple = lr_simple.predict_proba(X_test_scaled)[:, 1]
results_lr_simple = evaluate_model("Logistic Regression (baseline)", y_test, y_pred_lr_simple, y_proba_lr_simple)


Model: Logistic Regression (baseline)
              precision    recall  f1-score   support

         Low       0.89      0.68      0.77     24141
        High       0.33      0.67      0.44      5859

    accuracy                           0.67     30000
   macro avg       0.61      0.67      0.61     30000
weighted avg       0.78      0.67      0.71     30000

Confusion Matrix:
[[16322  7819]
 [ 1960  3899]]
Accuracy: 0.6740
ROC-AUC: 0.7243


F1 Scorer for Hyperparameter Tuning


In [4]:
f1_scorer = make_scorer(f1_score, pos_label=1)

### 5.2 Random Forest

Random Forest with hyperparameter tuning via RandomizedSearchCV, optimized for F1 (High). target.

In [5]:
# Random Forest with hyperparameter tuning (F1) on scaled data
rf_param_grid = {"n_estimators": [80, 120, 150], "max_depth": [12, 15, 20], "min_samples_leaf": [2, 4, 6]}
rf_search = RandomizedSearchCV(RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1), param_distributions=rf_param_grid, n_iter=30, scoring=f1_scorer, cv=3, random_state=42, n_jobs=-1)
rf_search.fit(X_train_scaled, y_train)
rf = rf_search.best_estimator_
y_pred_rf = rf.predict(X_test_scaled)
y_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]
results_rf = evaluate_model("Random Forest", y_test, y_pred_rf, y_proba_rf)
print("Best RF params:", rf_search.best_params_)

c:\Users\nitin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 27 is smaller than n_iter=30. Running 27 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\nitin\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\ma\core.py:2881: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,



Model: Random Forest
              precision    recall  f1-score   support

         Low       0.91      0.78      0.84     24141
        High       0.43      0.69      0.53      5859

    accuracy                           0.76     30000
   macro avg       0.67      0.73      0.68     30000
weighted avg       0.82      0.76      0.78     30000

Confusion Matrix:
[[18757  5384]
 [ 1837  4022]]
Accuracy: 0.7593
ROC-AUC: 0.8092
Best RF params: {'n_estimators': 120, 'min_samples_leaf': 4, 'max_depth': 20}


### 5.3 XgBoost with Hyperparameter Tuning (F1) on scaled data

In [6]:

scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
xgb_param_grid = {"n_estimators": [80, 120, 150], "max_depth": [4, 6, 8], "learning_rate": [0.05, 0.1, 0.15], "min_child_weight": [1, 3, 5]}
xgb_search = RandomizedSearchCV(xgb.XGBClassifier(random_state=42, scale_pos_weight=scale_pos), param_distributions=xgb_param_grid, n_iter=30, scoring=f1_scorer, cv=3, random_state=42, n_jobs=-1)
xgb_search.fit(X_train_scaled, y_train)
xgb_clf = xgb_search.best_estimator_
y_pred_xgb = xgb_clf.predict(X_test_scaled)
y_proba_xgb = xgb_clf.predict_proba(X_test_scaled)[:, 1]

results_xgb = evaluate_model("XGBoost", y_test, y_pred_xgb, y_proba_xgb)
print("Best XGB params:", xgb_search.best_params_)


Model: XGBoost
              precision    recall  f1-score   support

         Low       0.92      0.76      0.83     24141
        High       0.42      0.74      0.54      5859

    accuracy                           0.75     30000
   macro avg       0.67      0.75      0.68     30000
weighted avg       0.82      0.75      0.77     30000

Confusion Matrix:
[[18235  5906]
 [ 1537  4322]]
Accuracy: 0.7519
ROC-AUC: 0.8222
Best XGB params: {'n_estimators': 150, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.1}


### 5.4 DNN (Deep Neural Network)

Multi-layer perceptron with dropout and class weighting for the imbalanced target.ty rate varies by hour of day and by infrastructure flags.

In [7]:
# DNN (MLP) on scaled data with class weights for imbalance
keras.utils.set_random_seed(42)
n_features = X_train_scaled.shape[1]
class_weight = {0: 1.0, 1: (y_train == 0).sum() / max((y_train == 1).sum(), 1)}
dnn = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(n_features,)),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
dnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
dnn.fit(X_train_scaled, y_train, epochs=10, batch_size=512, class_weight=class_weight, validation_split=0.1, verbose=0)
y_proba_dnn = dnn.predict(X_test_scaled, verbose=0).ravel()
y_pred_dnn = (y_proba_dnn >= 0.5).astype(int)
results_dnn = evaluate_model("DNN", y_test, y_pred_dnn, y_proba_dnn)

c:\Users\nitin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Model: DNN
              precision    recall  f1-score   support

         Low       0.91      0.67      0.77     24141
        High       0.34      0.72      0.46      5859

    accuracy                           0.68     30000
   macro avg       0.62      0.69      0.62     30000
weighted avg       0.80      0.68      0.71     30000

Confusion Matrix:
[[16087  8054]
 [ 1668  4191]]
Accuracy: 0.6759
ROC-AUC: 0.7566


### 5.5 Stacking Classifier

In [8]:
# Wrapper so Keras DNN can be used in StackingClassifier
class KerasClassifierWrapper:
    def __init__(self, epochs=10, batch_size=512, random_state=42):
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.model_ = None

    def fit(self, X, y):
        keras.utils.set_random_seed(self.random_state)
        n_features = X.shape[1]
        cw = {0: 1.0, 1: (y == 0).sum() / max((y == 1).sum(), 1)}
        self.model_ = keras.Sequential([
            layers.Dense(64, activation="relu", input_shape=(n_features,)),
            layers.Dropout(0.3),
            layers.Dense(32, activation="relu"),
            layers.Dropout(0.2),
            layers.Dense(1, activation="sigmoid"),
        ])
        self.model_.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        self.model_.fit(X, y, epochs=self.epochs, batch_size=self.batch_size, class_weight=cw, validation_split=0.1, verbose=0)
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        p = self.model_.predict(X, verbose=0).ravel()
        return (p >= 0.5).astype(int)

    def predict_proba(self, X):
        p = self.model_.predict(X, verbose=0).ravel()
        return np.column_stack([1 - p, p])

    def get_params(self, deep=True):
        return {"epochs": self.epochs, "batch_size": self.batch_size, "random_state": self.random_state}

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

dnn_estimator = KerasClassifierWrapper(epochs=10, batch_size=512, random_state=42)

In [9]:
# Stacking ensemble: LR + RF + XGBoost + DNN with logistic regression meta-learner
stack = StackingClassifier(
    estimators=[
        ("lr", lr_simple),
        ("rf", rf),
        ("xgb", xgb_clf),
        ("dnn", dnn_estimator),
    ],
    final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    cv=3,
)
stack.fit(X_train_scaled, y_train)
y_pred_stack = stack.predict(X_test_scaled)
y_proba_stack = stack.predict_proba(X_test_scaled)[:, 1]
results_stack = evaluate_model("Stacking (LR+RF+XGB+DNN)", y_test, y_pred_stack, y_proba_stack)

c:\Users\nitin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\nitin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Model: Stacking (LR+RF+XGB+DNN)
              precision    recall  f1-score   support

         Low       0.93      0.73      0.82     24141
        High       0.41      0.76      0.53      5859

    accuracy                           0.74     30000
   macro avg       0.67      0.75      0.68     30000
weighted avg       0.83      0.74      0.76     30000

Confusion Matrix:
[[17703  6438]
 [ 1390  4469]]
Accuracy: 0.7391
ROC-AUC: 0.8223


In [10]:
from pathlib import Path
import joblib

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

training_artifacts = {}
for _name in [
    "results_lr_simple",
    "results_rf",
    "results_xgb",
    "results_dnn",
    "results_stack",
    "rf",
    "xgb_search",
]:
    if _name in globals():
        training_artifacts[_name] = globals()[_name]

joblib.dump(training_artifacts, artifacts_dir / "training_artifacts.joblib")
print("Saved training artifacts to", artifacts_dir / "training_artifacts.joblib")


Saved training artifacts to artifacts\training_artifacts.joblib
